# 07. Downside-aware +3% 직접 target

요청한 개선을 한 번에 섞지 않고 첫 번째 항목만 검증한다. `06_modern_tcn_baseline.ipynb`의 데이터, 60봉 입력, context, 날짜 OOF fold, stride=3, scaler, ModernTCN 구조, optimizer, loss와 epoch 후보를 모두 유지하고 **target만 변경**한다.

## Target

```text
original TP:       MFE_3m >= +3%
downside-aware TP: MFE_3m >= +3% AND downside_3m < 3%
```

`downside = -MAE`다. 같은 3분 동안 위·아래가 모두 3% 이상 움직인 샘플은 고변동 양방향 움직임으로 보고 negative 처리한다. 새로운 손절 기준을 튜닝하지 않고 기존 3% 하나만 사용한다.

1분 OHLC만으로 같은 봉 안에서 high와 low 중 어느 것이 먼저 발생했는지는 알 수 없다. 따라서 이 target은 TP-first 체결 라벨이 아니라 **보수적인 clean-upward-excursion 라벨**이다.

비교는 기존 ModernTCN과 새 target 모델을 동일한 downside-aware 정답에서 평가한다. 아직 utility 직접 학습, episode 평가, percentile threshold, 신규 context는 적용하지 않는다.

In [ ]:
from pathlib import Path
import json
import math
import random
import time
from copy import deepcopy

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    mean_absolute_error,
    roc_auc_score,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENT.md').is_file() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')


PROJECT_ROOT = find_project_root()
PREPROCESS_ROOT = (PROJECT_ROOT / '../../results/preprocessing').resolve()
MODEL_ROOT = (PROJECT_ROOT / '../../model/moderntcn_downside_aware_tp3_v1').resolve()
RESULT_ROOT = (PROJECT_ROOT / '../../results/training/moderntcn_downside_aware_tp3_v1').resolve()
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

DATA_VERSION = 'ohlc_60m_tp3pct_v1'
RANDOM_VERSION = 'ohlc_60m_tp3pct_random_entry_v1'
SEED = 20260721
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CONFIG = {
    'sequence_length': 60,
    'input_features': 11,
    'context_features': 20,
    'd_model': 16,
    'd_ff': 48,
    'num_blocks': 2,
    'patch_size': 8,
    'patch_stride': 4,
    'large_kernel_size': 7,
    'small_kernel_size': 3,
    'dropout': 0.15,
    'batch_size': 512,
    'candidate_epochs': [4, 8, 12],
    'learning_rate': 3e-4,
    'weight_decay': 1e-3,
    'gradient_clip': 1.0,
    'train_stride_minutes': 3,
    'input_clip': 10.0,
    'regression_quantile_clip': 0.99,
    'smooth_l1_beta': 0.5,
    'num_workers': 0,
}


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)
torch.set_float32_matmul_precision('high')

print('project:', PROJECT_ROOT)
print('device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)
print('model output:', MODEL_ROOT)
print('result output:', RESULT_ROOT)

## 1. 전처리 artifact와 날짜 split 로드

NPZ 배열 순서와 metadata / random-label / split의 `sample_id`가 완전히 같은지 검사한다. scaling은 저장 데이터에 미리 적용하지 않고 각 OOF fit 날짜에서만 계산한다.

In [ ]:
schema = json.loads((PREPROCESS_ROOT / f'{DATA_VERSION}_schema.json').read_text(encoding='utf-8'))
fold_document = json.loads((PREPROCESS_ROOT / f'{DATA_VERSION}_walk_forward_folds.json').read_text(encoding='utf-8'))

metadata = pd.read_parquet(PREPROCESS_ROOT / f'{DATA_VERSION}_metadata.parquet')
random_labels = pd.read_parquet(PREPROCESS_ROOT / f'{RANDOM_VERSION}_metadata.parquet')
split = pd.read_parquet(PREPROCESS_ROOT / f'{DATA_VERSION}_split.parquet')

with np.load(PREPROCESS_ROOT / f'{DATA_VERSION}_features.npz') as arrays:
    sequence = arrays['sequence'].astype(np.float32, copy=False)
    context = arrays['context'].astype(np.float32, copy=False)

assert metadata['sample_id'].is_unique
assert metadata['sample_id'].tolist() == random_labels['sample_id'].tolist() == split['sample_id'].tolist()
assert len(metadata) == len(sequence) == len(context)
assert sequence.shape[1:] == (CONFIG['sequence_length'], CONFIG['input_features'])
assert context.shape[1] == CONFIG['context_features']
assert np.isfinite(sequence).all() and np.isfinite(context).all()
assert fold_document['test_is_pristine'] is False

y_original_hard = metadata['target_tp3_3m'].to_numpy(np.float32)
downside_3m = (-metadata['mae_3m']).to_numpy(np.float32)
both_sides_3pct = (y_original_hard == 1.0) & (downside_3m >= 0.03)
y_hard = ((y_original_hard == 1.0) & (downside_3m < 0.03)).astype(np.float32)
y_soft = random_labels['p_tp3_random_entry_3m'].to_numpy(np.float32)
y_reg = np.column_stack([
    metadata['mfe_3m'].to_numpy(np.float32),
    -metadata['mae_3m'].to_numpy(np.float32),
]).astype(np.float32)

assert np.isin(y_hard, [0.0, 1.0]).all()
assert ((0 <= y_soft) & (y_soft <= 1)).all()
assert (y_reg >= -1e-7).all()

# 한 run 안에서 매분 생성된 60봉 window의 중복을 줄인다. OOF/Test 평가는 모든 시점에서 한다.
within_run_position = metadata.groupby('run_id', sort=False).cumcount().to_numpy()
train_sampling_mask = (within_run_position % CONFIG['train_stride_minutes']) == 0

train_mask = split['split'].eq('train').to_numpy()
test_mask = split['split'].eq('test').to_numpy()
oof_mask = split['oof_fold'].gt(0).to_numpy()

dataset_summary = pd.DataFrame({
    'partition': ['Train 전체', 'Train 학습 stride=3', 'OOF 평가', 'Test 평가'],
    'samples': [
        int(train_mask.sum()),
        int((train_mask & train_sampling_mask).sum()),
        int(oof_mask.sum()),
        int(test_mask.sum()),
    ],
})
display(dataset_summary)
display(pd.DataFrame({
    'target': ['downside-aware TP3/3m', 'original TP3/3m'],
    'Train mean': [float(y_hard[train_mask].mean()), float(y_original_hard[train_mask].mean())],
    'Test mean': [float(y_hard[test_mask].mean()), float(y_original_hard[test_mask].mean())],
}).style.format({'Train mean': '{:.3%}', 'Test mean': '{:.3%}'}))
print('sequence:', sequence.shape, 'context:', context.shape)
print('OOF folds:', len(fold_document['folds']))

target_change = pd.DataFrame([
    {
        'partition': name,
        'samples': int(mask.sum()),
        'original_positives': int(y_original_hard[mask].sum()),
        'downside_aware_positives': int(y_hard[mask].sum()),
        'both_sides_relabelled_negative': int(both_sides_3pct[mask].sum()),
        'removed_positive_share': float(
            both_sides_3pct[mask].sum() / max(y_original_hard[mask].sum(), 1)
        ),
    }
    for name, mask in [('Train', train_mask), ('OOF', oof_mask), ('Test', test_mask)]
])
display(target_change.style.format({'removed_positive_share': '{:.2%}'}))

## 2. 고정된 Compact ModernTCN 백본

06번과 동일한 117,825 parameter Compact ModernTCN을 그대로 사용한다. 이 실험에서 바뀌는 것은 `target_tp3_3m`을 downside-aware target으로 교체한 것뿐이다.

In [ ]:
class ConvBN(nn.Module):
    def __init__(self, channels, kernel_size):
        super().__init__()
        self.conv = nn.Conv1d(
            channels,
            channels,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            groups=channels,
            bias=False,
        )
        self.bn = nn.BatchNorm1d(channels)

    def forward(self, x):
        return self.bn(self.conv(x))


class ReparamLargeSmallKernel(nn.Module):
    def __init__(self, channels, large_kernel, small_kernel):
        super().__init__()
        self.large_branch = ConvBN(channels, large_kernel)
        self.small_branch = ConvBN(channels, small_kernel)

    def forward(self, x):
        return self.large_branch(x) + self.small_branch(x)


class ModernTCNBlock(nn.Module):
    def __init__(
        self,
        n_variables,
        d_model,
        d_ff,
        large_kernel,
        small_kernel,
        dropout,
    ):
        super().__init__()
        channels = n_variables * d_model
        self.n_variables = n_variables
        self.d_model = d_model
        self.d_ff = d_ff

        self.temporal_mixer = ReparamLargeSmallKernel(
            channels, large_kernel, small_kernel
        )
        self.feature_norm = nn.BatchNorm1d(d_model)

        # 각 변수 내부 embedding을 독립적으로 변환한다.
        self.within_variable_ffn = nn.Sequential(
            nn.Conv1d(channels, n_variables * d_ff, kernel_size=1, groups=n_variables),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(n_variables * d_ff, channels, kernel_size=1, groups=n_variables),
            nn.Dropout(dropout),
        )

        # embedding channel별로 변수 축을 섞는다.
        self.cross_variable_ffn = nn.Sequential(
            nn.Conv1d(channels, n_variables * d_ff, kernel_size=1, groups=d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(n_variables * d_ff, channels, kernel_size=1, groups=d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # x: [B, M, D, N]
        residual = x
        batch, variables, embedding, patches = x.shape

        x = self.temporal_mixer(x.reshape(batch, variables * embedding, patches))
        x = x.reshape(batch * variables, embedding, patches)
        x = self.feature_norm(x)
        x = x.reshape(batch, variables * embedding, patches)

        x = self.within_variable_ffn(x)
        x = x.reshape(batch, variables, embedding, patches)

        x = x.permute(0, 2, 1, 3).reshape(batch, embedding * variables, patches)
        x = self.cross_variable_ffn(x)
        x = x.reshape(batch, embedding, variables, patches).permute(0, 2, 1, 3)
        return residual + x


class CompactModernTCN(nn.Module):
    def __init__(
        self,
        input_features,
        context_features,
        output_dim,
        d_model=16,
        d_ff=48,
        num_blocks=2,
        patch_size=8,
        patch_stride=4,
        large_kernel_size=7,
        small_kernel_size=3,
        dropout=0.15,
    ):
        super().__init__()
        self.input_features = input_features
        self.patch_size = patch_size
        self.patch_stride = patch_stride
        self.patch_embedding = nn.Sequential(
            nn.Conv1d(1, d_model, kernel_size=patch_size, stride=patch_stride),
            nn.BatchNorm1d(d_model),
        )
        self.blocks = nn.ModuleList([
            ModernTCNBlock(
                input_features,
                d_model,
                d_ff,
                large_kernel_size,
                small_kernel_size,
                dropout,
            )
            for _ in range(num_blocks)
        ])
        self.output_norm = nn.BatchNorm1d(d_model)
        self.context_encoder = nn.Sequential(
            nn.Linear(context_features, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        pooled_dim = input_features * d_model * 3 + d_model
        self.head = nn.Sequential(
            nn.Linear(pooled_dim, d_ff * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff * 2, output_dim),
        )
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Conv1d)):
            if module.weight is not None:
                nn.init.kaiming_normal_(module.weight, nonlinearity='linear')
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, sequence_x, context_x):
        # [B, T, M] -> 변수별 patch embedding -> [B, M, D, N]
        x = sequence_x.transpose(1, 2)
        batch, variables, length = x.shape
        pad_length = self.patch_size - self.patch_stride
        if pad_length > 0:
            x = torch.cat([x, x[:, :, -1:].expand(-1, -1, pad_length)], dim=-1)
        x = self.patch_embedding(x.reshape(batch * variables, 1, -1))
        embedding, patches = x.shape[1], x.shape[2]
        x = x.reshape(batch, variables, embedding, patches)

        for block in self.blocks:
            x = block(x)

        x = self.output_norm(x.reshape(batch * variables, embedding, patches))
        x = x.reshape(batch, variables, embedding, patches)
        pooled = torch.cat([
            x.mean(dim=-1),
            x.amax(dim=-1),
            x[:, :, :, -1],
        ], dim=2).reshape(batch, -1)
        fused = torch.cat([pooled, self.context_encoder(context_x)], dim=1)
        return self.head(fused)


smoke_model = CompactModernTCN(
    input_features=CONFIG['input_features'],
    context_features=CONFIG['context_features'],
    output_dim=1,
    d_model=CONFIG['d_model'],
    d_ff=CONFIG['d_ff'],
    num_blocks=CONFIG['num_blocks'],
    patch_size=CONFIG['patch_size'],
    patch_stride=CONFIG['patch_stride'],
    large_kernel_size=CONFIG['large_kernel_size'],
    small_kernel_size=CONFIG['small_kernel_size'],
    dropout=CONFIG['dropout'],
).to(DEVICE)
with torch.inference_mode():
    smoke_output = smoke_model(
        torch.zeros(2, 60, 11, device=DEVICE),
        torch.zeros(2, 20, device=DEVICE),
    )
MODEL_PARAMETER_COUNT = sum(p.numel() for p in smoke_model.parameters())
print('parameters:', f'{MODEL_PARAMETER_COUNT:,}')
print('TimeMixer++ adaptation parameters: 130,999')
print('parameter ratio:', f'{MODEL_PARAMETER_COUNT / 130999:.3f}')
print('smoke output:', tuple(smoke_output.shape))
del smoke_model, smoke_output
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 3. Fold-local scaling, loss, metric

- 입력 median/IQR은 각 fold의 실제 학습 subset에서만 계산하고 ±10 IQR로 clip한다.
- 회귀 target도 fold Train의 q99와 median/IQR만 사용한다.
- BCE에는 class weight와 focal loss를 넣지 않는다. 먼저 확률 calibration을 보존한 기준 성능을 확인한다.
- threshold/체결 조건/백테스트는 이 학습 노트북의 범위가 아니다.

In [ ]:
STRATEGIES = {
    'downside_aware_hard': {'output_dim': 1, 'selection_metric': 'hard_pr_auc', 'mode': 'max'},
}


def robust_center_scale(values, axis=0):
    center = np.median(values, axis=axis).astype(np.float32)
    q25 = np.quantile(values, 0.25, axis=axis).astype(np.float32)
    q75 = np.quantile(values, 0.75, axis=axis).astype(np.float32)
    scale = (q75 - q25).astype(np.float32)
    scale = np.where(scale < 1e-6, 1.0, scale).astype(np.float32)
    return center, scale


def fit_input_scaler(indices):
    seq_values = sequence[indices].reshape(-1, sequence.shape[-1])
    seq_center, seq_scale = robust_center_scale(seq_values, axis=0)
    ctx_center, ctx_scale = robust_center_scale(context[indices], axis=0)
    return {
        'sequence_center': seq_center,
        'sequence_scale': seq_scale,
        'context_center': ctx_center,
        'context_scale': ctx_scale,
    }


def transform_inputs(indices, scaler):
    seq = (sequence[indices] - scaler['sequence_center'][None, None, :]) / scaler['sequence_scale'][None, None, :]
    ctx = (context[indices] - scaler['context_center'][None, :]) / scaler['context_scale'][None, :]
    seq = np.clip(seq, -CONFIG['input_clip'], CONFIG['input_clip']).astype(np.float32)
    ctx = np.clip(ctx, -CONFIG['input_clip'], CONFIG['input_clip']).astype(np.float32)
    return np.ascontiguousarray(seq), np.ascontiguousarray(ctx)


def fit_regression_scaler(indices):
    values = y_reg[indices]
    upper = np.quantile(values, CONFIG['regression_quantile_clip'], axis=0).astype(np.float32)
    clipped = np.minimum(values, upper)
    center, scale = robust_center_scale(clipped, axis=0)
    return {'upper': upper, 'center': center, 'scale': scale}


def transform_regression_target(indices, scaler):
    values = np.minimum(y_reg[indices], scaler['upper'])
    return ((values - scaler['center']) / scaler['scale']).astype(np.float32)


def inverse_regression_target(values, scaler):
    restored = values * scaler['scale'][None, :] + scaler['center'][None, :]
    return np.clip(restored, 0.0, scaler['upper'][None, :]).astype(np.float32)


def make_loader(indices, scaler, strategy, target_scaler=None, shuffle=False, seed=SEED):
    seq, ctx = transform_inputs(indices, scaler)
    if strategy == 'downside_aware_hard':
        target = y_hard[indices, None]
    elif strategy == 'random_soft':
        target = y_soft[indices, None]
    else:
        target = transform_regression_target(indices, target_scaler)
    dataset = TensorDataset(
        torch.from_numpy(seq),
        torch.from_numpy(ctx),
        torch.from_numpy(np.ascontiguousarray(target, dtype=np.float32)),
    )
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=shuffle,
        num_workers=CONFIG['num_workers'],
        pin_memory=DEVICE.type == 'cuda',
        drop_last=False,
        generator=generator,
    )


def build_model(strategy):
    return CompactModernTCN(
        input_features=CONFIG['input_features'],
        context_features=CONFIG['context_features'],
        output_dim=STRATEGIES[strategy]['output_dim'],
        d_model=CONFIG['d_model'],
        d_ff=CONFIG['d_ff'],
        num_blocks=CONFIG['num_blocks'],
        patch_size=CONFIG['patch_size'],
        patch_stride=CONFIG['patch_stride'],
        large_kernel_size=CONFIG['large_kernel_size'],
        small_kernel_size=CONFIG['small_kernel_size'],
        dropout=CONFIG['dropout'],
    ).to(DEVICE)


def loss_for(strategy, output, target):
    if strategy == 'downside_aware_hard':
        return F.binary_cross_entropy_with_logits(output, target)
    return F.smooth_l1_loss(output, target, beta=CONFIG['smooth_l1_beta'])


def predict(model, loader):
    model.eval()
    outputs = []
    with torch.inference_mode():
        for seq_batch, ctx_batch, _ in loader:
            seq_batch = seq_batch.to(DEVICE, non_blocking=True)
            ctx_batch = ctx_batch.to(DEVICE, non_blocking=True)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.bfloat16,
                enabled=DEVICE.type == 'cuda',
            ):
                output = model(seq_batch, ctx_batch)
            outputs.append(output.float().cpu().numpy())
    return np.concatenate(outputs)


def clipped_probability(logits):
    probabilities = 1.0 / (1.0 + np.exp(-np.clip(logits, -30, 30)))
    return np.clip(probabilities, 1e-7, 1 - 1e-7)


def soft_binary_cross_entropy(target, probability):
    return float(-np.mean(target * np.log(probability) + (1 - target) * np.log(1 - probability)))


def safe_spearman(actual, predicted):
    value = spearmanr(actual, predicted).statistic
    return float(value) if np.isfinite(value) else np.nan


def classification_metrics(indices, logits, strategy):
    probability = clipped_probability(logits.reshape(-1))
    hard = y_hard[indices]
    soft = y_soft[indices]
    return {
        'hard_pr_auc': float(average_precision_score(hard, probability)),
        'hard_roc_auc': float(roc_auc_score(hard, probability)) if np.unique(hard).size > 1 else np.nan,
        'hard_log_loss': float(log_loss(hard, probability, labels=[0, 1])),
        'hard_brier': float(brier_score_loss(hard, probability)),
        'soft_bce': soft_binary_cross_entropy(soft, probability),
        'soft_brier': float(np.mean((soft - probability) ** 2)),
        'soft_spearman': safe_spearman(soft, probability),
        'prediction_mean': float(probability.mean()),
        'hard_prevalence': float(hard.mean()),
        'soft_target_mean': float(soft.mean()),
    }


def regression_metrics(indices, scaled_prediction, target_scaler):
    prediction = inverse_regression_target(scaled_prediction, target_scaler)
    actual = y_reg[indices]
    score_pred = prediction[:, 0] - prediction[:, 1]
    score_actual = actual[:, 0] - actual[:, 1]
    return {
        'mfe_spearman': safe_spearman(actual[:, 0], prediction[:, 0]),
        'downside_spearman': safe_spearman(actual[:, 1], prediction[:, 1]),
        'mean_target_spearman': float(np.nanmean([
            safe_spearman(actual[:, 0], prediction[:, 0]),
            safe_spearman(actual[:, 1], prediction[:, 1]),
        ])),
        'mfe_mae': float(mean_absolute_error(actual[:, 0], prediction[:, 0])),
        'downside_mae': float(mean_absolute_error(actual[:, 1], prediction[:, 1])),
        'utility_spearman': safe_spearman(score_actual, score_pred),
        'utility_hard_pr_auc': float(average_precision_score(y_hard[indices], score_pred)),
        'predicted_mfe_mean': float(prediction[:, 0].mean()),
        'predicted_downside_mean': float(prediction[:, 1].mean()),
    }


def evaluate_predictions(indices, raw_prediction, strategy, target_scaler=None):
    if strategy == 'downside_aware_hard':
        return classification_metrics(indices, raw_prediction, strategy)
    return regression_metrics(indices, raw_prediction, target_scaler)


def train_epochs(strategy, fit_indices, eval_indices, candidate_epochs, run_seed):
    seed_everything(run_seed)
    input_scaler = fit_input_scaler(fit_indices)
    target_scaler = fit_regression_scaler(fit_indices) if strategy == 'mfe_mae' else None
    train_loader = make_loader(
        fit_indices, input_scaler, strategy, target_scaler, shuffle=True, seed=run_seed
    )
    eval_loader = make_loader(
        eval_indices, input_scaler, strategy, target_scaler, shuffle=False, seed=run_seed
    )
    model = build_model(strategy)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CONFIG['learning_rate'],
        weight_decay=CONFIG['weight_decay'],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(candidate_epochs), eta_min=CONFIG['learning_rate'] * 0.1
    )
    history = []
    snapshots = {}
    predictions = {}

    for epoch in range(1, max(candidate_epochs) + 1):
        model.train()
        total_loss = 0.0
        total_count = 0
        for seq_batch, ctx_batch, target_batch in train_loader:
            seq_batch = seq_batch.to(DEVICE, non_blocking=True)
            ctx_batch = ctx_batch.to(DEVICE, non_blocking=True)
            target_batch = target_batch.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.bfloat16,
                enabled=DEVICE.type == 'cuda',
            ):
                output = model(seq_batch, ctx_batch)
                loss = loss_for(strategy, output, target_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CONFIG['gradient_clip'])
            optimizer.step()
            total_loss += float(loss.detach()) * len(target_batch)
            total_count += len(target_batch)
        scheduler.step()
        history.append({
            'epoch': epoch,
            'train_loss': total_loss / total_count,
            'learning_rate': optimizer.param_groups[0]['lr'],
        })
        if epoch in candidate_epochs:
            raw_prediction = predict(model, eval_loader)
            predictions[epoch] = raw_prediction
            snapshots[epoch] = deepcopy(model.state_dict())

    del train_loader, eval_loader, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {
        'predictions': predictions,
        'snapshots': snapshots,
        'history': history,
        'input_scaler': input_scaler,
        'target_scaler': target_scaler,
    }

## 4. Downside-aware target의 expanding walk-forward OOF 학습

과거 날짜만 fit하고 다음 한 날짜를 예측한다. 모델 구조와 학습 설정은 06번과 같으며 target만 달라진다. Test는 epoch 선택에 사용하지 않는다.

In [ ]:
fold_records = []
history_records = []
oof_raw_predictions = {
    strategy: {epoch: np.full((len(metadata), spec['output_dim']), np.nan, dtype=np.float32)
               for epoch in CONFIG['candidate_epochs']}
    for strategy, spec in STRATEGIES.items()
}
oof_target_scalers = {strategy: {} for strategy in STRATEGIES}

oof_start = time.time()
for strategy_index, strategy in enumerate(STRATEGIES):
    print(f'\n===== OOF strategy: {strategy} =====')
    for fold in fold_document['folds']:
        fold_number = int(fold['fold'])
        fit_all = np.flatnonzero(split['session'].isin(fold['fit_sessions']).to_numpy())
        fit_indices = fit_all[train_sampling_mask[fit_all]]
        eval_indices = np.flatnonzero(split['session'].eq(fold['evaluation_session']).to_numpy())
        assert split.iloc[fit_indices]['decision_timestamp'].max() < split.iloc[eval_indices]['decision_timestamp'].min()

        run_seed = SEED + strategy_index * 100 + fold_number
        result = train_epochs(
            strategy,
            fit_indices,
            eval_indices,
            CONFIG['candidate_epochs'],
            run_seed,
        )
        oof_target_scalers[strategy][fold_number] = result['target_scaler']

        for row in result['history']:
            history_records.append({
                'phase': 'oof', 'strategy': strategy, 'fold': fold_number,
                'fit_sessions': ','.join(fold['fit_sessions']),
                'evaluation_session': fold['evaluation_session'],
                'fit_samples_after_stride': len(fit_indices), **row,
            })

        for epoch, raw_prediction in result['predictions'].items():
            oof_raw_predictions[strategy][epoch][eval_indices] = raw_prediction
            metrics = evaluate_predictions(
                eval_indices, raw_prediction, strategy, result['target_scaler']
            )
            fold_records.append({
                'strategy': strategy,
                'fold': fold_number,
                'evaluation_session': fold['evaluation_session'],
                'epoch': epoch,
                'fit_samples_after_stride': len(fit_indices),
                'evaluation_samples': len(eval_indices),
                **metrics,
            })
        primary = STRATEGIES[strategy]['selection_metric']
        last_value = fold_records[-1][primary]
        print(
            f"fold {fold_number} | fit {len(fit_indices):,} | eval {len(eval_indices):,} "
            f"| epoch {max(CONFIG['candidate_epochs'])} {primary}={last_value:.5f}"
        )

print(f'\nOOF elapsed: {(time.time() - oof_start) / 60:.2f} min')
fold_metrics = pd.DataFrame(fold_records)
training_history = pd.DataFrame(history_records)
display(fold_metrics.head())

## 5. OOF로 epoch 선택

네 OOF 날짜의 downside-aware 예측을 이어 붙인 pooled PR-AUC로 epoch를 선택한다. 기존 모델과의 비교 역시 새 정답을 기준으로 다시 계산한다.

In [ ]:
epoch_records = []
selected_epochs = {}

oof_indices = np.flatnonzero(oof_mask)
for strategy, spec in STRATEGIES.items():
    for epoch in CONFIG['candidate_epochs']:
        raw = oof_raw_predictions[strategy][epoch][oof_indices]
        assert np.isfinite(raw).all()
        if strategy == 'mfe_mae':
            # 회귀 prediction은 fold별 target scaling이 다르므로 원 단위로 복원해 합친다.
            restored = np.full_like(raw, np.nan)
            for fold in fold_document['folds']:
                fold_number = int(fold['fold'])
                local_mask = split.iloc[oof_indices]['oof_fold'].to_numpy() == fold_number
                restored[local_mask] = inverse_regression_target(
                    raw[local_mask], oof_target_scalers[strategy][fold_number]
                )
            actual = y_reg[oof_indices]
            mfe_s = safe_spearman(actual[:, 0], restored[:, 0])
            down_s = safe_spearman(actual[:, 1], restored[:, 1])
            score_pred = restored[:, 0] - restored[:, 1]
            pooled = {
                'mfe_spearman': mfe_s,
                'downside_spearman': down_s,
                'mean_target_spearman': float(np.nanmean([mfe_s, down_s])),
                'mfe_mae': float(mean_absolute_error(actual[:, 0], restored[:, 0])),
                'downside_mae': float(mean_absolute_error(actual[:, 1], restored[:, 1])),
                'utility_spearman': safe_spearman(actual[:, 0] - actual[:, 1], score_pred),
                'utility_hard_pr_auc': float(average_precision_score(y_hard[oof_indices], score_pred)),
            }
        else:
            pooled = classification_metrics(oof_indices, raw, strategy)

        candidate_fold_rows = fold_metrics[
            fold_metrics['strategy'].eq(strategy) & fold_metrics['epoch'].eq(epoch)
        ]
        primary = spec['selection_metric']
        epoch_records.append({
            'strategy': strategy,
            'epoch': epoch,
            **pooled,
            f'worst_daily_{primary}': (
                float(candidate_fold_rows[primary].min())
                if spec['mode'] == 'max'
                else float(candidate_fold_rows[primary].max())
            ),
        })

epoch_selection = pd.DataFrame(epoch_records)
for strategy, spec in STRATEGIES.items():
    rows = epoch_selection[epoch_selection['strategy'].eq(strategy)]
    metric = spec['selection_metric']
    best_index = rows[metric].idxmax() if spec['mode'] == 'max' else rows[metric].idxmin()
    selected_epochs[strategy] = int(epoch_selection.loc[best_index, 'epoch'])

selection_view_columns = [
    'strategy', 'epoch', 'hard_pr_auc', 'soft_bce',
    'mfe_spearman', 'downside_spearman', 'mean_target_spearman',
]
selection_view_columns = [c for c in selection_view_columns if c in epoch_selection]
display(epoch_selection[selection_view_columns].sort_values(['strategy', 'epoch']))
print('selected epochs:', selected_epochs)

## 6. 전체 Train 재학습과 고정 Test 진단

OOF에서 선택된 epoch로 7개 Train 날짜 전체를 다시 학습한다. Test는 여기서 처음 평가한다.

In [ ]:
final_models = {}
final_input_scalers = {}
final_target_scalers = {}
test_raw_predictions = {}
test_metric_records = []

full_train_all = np.flatnonzero(train_mask)
full_train_indices = full_train_all[train_sampling_mask[full_train_all]]
test_indices = np.flatnonzero(test_mask)


def train_final_model(strategy, fit_indices, eval_indices, selected_epoch, run_seed):
    seed_everything(run_seed)
    input_scaler = fit_input_scaler(fit_indices)
    target_scaler = fit_regression_scaler(fit_indices) if strategy == 'mfe_mae' else None
    train_loader = make_loader(
        fit_indices, input_scaler, strategy, target_scaler, shuffle=True, seed=run_seed
    )
    eval_loader = make_loader(
        eval_indices, input_scaler, strategy, target_scaler, shuffle=False, seed=run_seed
    )
    model = build_model(strategy)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay']
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max(CONFIG['candidate_epochs']),
        eta_min=CONFIG['learning_rate'] * 0.1,
    )
    history = []
    for epoch in range(1, selected_epoch + 1):
        model.train()
        total_loss = 0.0
        total_count = 0
        for seq_batch, ctx_batch, target_batch in train_loader:
            seq_batch = seq_batch.to(DEVICE, non_blocking=True)
            ctx_batch = ctx_batch.to(DEVICE, non_blocking=True)
            target_batch = target_batch.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.bfloat16,
                enabled=DEVICE.type == 'cuda',
            ):
                output = model(seq_batch, ctx_batch)
                loss = loss_for(strategy, output, target_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CONFIG['gradient_clip'])
            optimizer.step()
            total_loss += float(loss.detach()) * len(target_batch)
            total_count += len(target_batch)
        scheduler.step()
        history.append({
            'phase': 'final', 'strategy': strategy, 'fold': 0,
            'fit_sessions': ','.join(fold_document['train_sessions']),
            'evaluation_session': ','.join(fold_document['test_sessions']),
            'fit_samples_after_stride': len(fit_indices),
            'epoch': epoch, 'train_loss': total_loss / total_count,
            'learning_rate': optimizer.param_groups[0]['lr'],
        })
    raw_prediction = predict(model, eval_loader)
    return model, input_scaler, target_scaler, raw_prediction, history


final_start = time.time()
for strategy_index, strategy in enumerate(STRATEGIES):
    chosen_epoch = selected_epochs[strategy]
    print(f'final {strategy}: {chosen_epoch} epochs')
    model, input_scaler, target_scaler, raw_prediction, history = train_final_model(
        strategy,
        full_train_indices,
        test_indices,
        chosen_epoch,
        SEED + 1000 + strategy_index,
    )
    final_models[strategy] = model
    final_input_scalers[strategy] = input_scaler
    final_target_scalers[strategy] = target_scaler
    test_raw_predictions[strategy] = raw_prediction
    history_records.extend(history)

    overall_metrics = evaluate_predictions(
        test_indices, raw_prediction, strategy, target_scaler
    )
    test_metric_records.append({
        'strategy': strategy, 'scope': 'all_test', 'session': 'ALL',
        'selected_epoch': chosen_epoch, 'samples': len(test_indices), **overall_metrics,
    })
    for session in fold_document['test_sessions']:
        local = np.flatnonzero(split['session'].eq(session).to_numpy())
        positions = np.searchsorted(test_indices, local)
        local_prediction = raw_prediction[positions]
        local_metrics = evaluate_predictions(local, local_prediction, strategy, target_scaler)
        test_metric_records.append({
            'strategy': strategy, 'scope': 'daily', 'session': session,
            'selected_epoch': chosen_epoch, 'samples': len(local), **local_metrics,
        })

print(f'final elapsed: {(time.time() - final_start) / 60:.2f} min')
test_metrics = pd.DataFrame(test_metric_records)
training_history = pd.DataFrame(history_records)

test_view_columns = [
    'strategy', 'scope', 'session', 'samples', 'selected_epoch',
    'hard_pr_auc', 'soft_bce', 'mfe_spearman', 'downside_spearman',
    'mean_target_spearman', 'utility_hard_pr_auc',
]
test_view_columns = [c for c in test_view_columns if c in test_metrics]
display(test_metrics[test_view_columns])

## 7. Target·모델·OOF/Test artifact 저장

새 target metadata와 모델 checkpoint를 별도 version으로 저장한다. 기존 preprocessing과 ModernTCN 결과는 덮어쓰지 않는다.

In [ ]:
def scaler_to_serializable(scaler):
    if scaler is None:
        return None
    return {key: np.asarray(value).tolist() for key, value in scaler.items()}


def make_prediction_frame(indices, phase):
    frame = metadata.iloc[indices][[
        'sample_id', 'session', 'symbol', 'decision_timestamp', 'entry_timestamp',
        'target_tp3_3m', 'mfe_3m', 'mae_3m',
    ]].reset_index(drop=True).copy()
    frame = frame.rename(columns={'target_tp3_3m': 'target_original_tp3_3m'})
    frame['target_downside_aware_tp3_3m'] = y_hard[indices].astype(np.int8)
    frame['both_sides_3pct'] = both_sides_3pct[indices]
    frame['downside_3m'] = downside_3m[indices]
    if phase == 'oof':
        frame['oof_fold'] = split.iloc[indices]['oof_fold'].to_numpy()
        raw = oof_raw_predictions['downside_aware_hard'][
            selected_epochs['downside_aware_hard']
        ][indices]
    else:
        raw = test_raw_predictions['downside_aware_hard']
    frame['pred_downside_aware_probability'] = clipped_probability(
        raw.reshape(-1)
    ).astype(np.float32)
    return frame


oof_predictions = make_prediction_frame(oof_indices, 'oof')
test_predictions = make_prediction_frame(test_indices, 'test')

target_metadata = metadata[[
    'sample_id', 'session', 'symbol', 'decision_timestamp', 'entry_timestamp'
]].copy()
target_metadata['target_original_tp3_3m'] = y_original_hard.astype(np.int8)
target_metadata['downside_3m'] = downside_3m
target_metadata['both_sides_3pct'] = both_sides_3pct
target_metadata['target_downside_aware_tp3_3m'] = y_hard.astype(np.int8)

target_path = PREPROCESS_ROOT / 'ohlc_60m_downside_aware_tp3_v1_metadata.parquet'
target_schema_path = PREPROCESS_ROOT / 'ohlc_60m_downside_aware_tp3_v1_schema.json'
target_metadata.to_parquet(target_path, index=False, compression='zstd')
target_schema = {
    'version': 'ohlc_60m_downside_aware_tp3_v1',
    'base_version': DATA_VERSION,
    'horizon_minutes': 3,
    'upside_threshold': 0.03,
    'downside_threshold': 0.03,
    'target_rule': 'target_tp3_3m == 1 and -mae_3m < 0.03',
    'both_sides_rule': 'MFE >= 3% and downside >= 3% is negative',
    'intrabar_order_known': False,
    'changed_from_previous_experiment': 'target only',
}
target_schema_path.write_text(
    json.dumps(target_schema, ensure_ascii=False, indent=2), encoding='utf-8'
)

model_paths = {}
for strategy, model in final_models.items():
    path = MODEL_ROOT / f'{strategy}.pt'
    checkpoint = {
        'model_class': 'CompactModernTCN',
        'implementation_scope': 'same CompactModernTCN as notebook 06; target-only ablation',
        'strategy': strategy,
        'selected_epoch': selected_epochs[strategy],
        'model_config': {
            key: CONFIG[key]
            for key in [
                'input_features', 'context_features', 'd_model', 'd_ff', 'num_blocks',
                'patch_size', 'patch_stride', 'large_kernel_size',
                'small_kernel_size', 'dropout',
            ]
        },
        'output_dim': 1,
        'state_dict': {key: value.detach().cpu() for key, value in model.state_dict().items()},
        'input_scaler': scaler_to_serializable(final_input_scalers[strategy]),
        'target_scaler': None,
        'target_definition': target_schema,
        'sequence_features': schema['sequence_features'],
        'context_features': schema['context_features'],
        'data_version': DATA_VERSION,
        'train_sessions': fold_document['train_sessions'],
        'test_sessions': fold_document['test_sessions'],
        'test_is_pristine': False,
        'seed': SEED,
        'parameter_count': MODEL_PARAMETER_COUNT,
    }
    torch.save(checkpoint, path)
    model_paths[strategy] = path

artifact_paths = {
    'fold_metrics': RESULT_ROOT / 'oof_daily_metrics.parquet',
    'epoch_selection': RESULT_ROOT / 'oof_epoch_selection.parquet',
    'test_metrics': RESULT_ROOT / 'test_metrics.parquet',
    'training_history': RESULT_ROOT / 'training_history.parquet',
    'oof_predictions': RESULT_ROOT / 'oof_predictions.parquet',
    'test_predictions': RESULT_ROOT / 'test_predictions.parquet',
    'manifest': RESULT_ROOT / 'manifest.json',
}

fold_metrics.to_parquet(artifact_paths['fold_metrics'], index=False, compression='zstd')
epoch_selection.to_parquet(artifact_paths['epoch_selection'], index=False, compression='zstd')
test_metrics.to_parquet(artifact_paths['test_metrics'], index=False, compression='zstd')
training_history.to_parquet(artifact_paths['training_history'], index=False, compression='zstd')
oof_predictions.to_parquet(artifact_paths['oof_predictions'], index=False, compression='zstd')
test_predictions.to_parquet(artifact_paths['test_predictions'], index=False, compression='zstd')

manifest = {
    'experiment': 'moderntcn_downside_aware_tp3_v1',
    'created_by_notebook': 'notebooks/07_downside_aware_target.ipynb',
    'ablation': 'target only',
    'baseline_experiment': 'moderntcn_ohlc_60m_v1',
    'data_version': DATA_VERSION,
    'target_version': target_schema['version'],
    'device': str(DEVICE),
    'torch_version': torch.__version__,
    'seed': SEED,
    'parameter_count': MODEL_PARAMETER_COUNT,
    'config': CONFIG,
    'selected_epochs': selected_epochs,
    'selection_data': 'Train expanding walk-forward OOF only',
    'test_role': fold_document['test_role'],
    'test_is_pristine': False,
    'model_paths': {key: str(value) for key, value in model_paths.items()},
    'result_paths': {key: str(value) for key, value in artifact_paths.items()},
    'target_paths': {
        'metadata': str(target_path), 'schema': str(target_schema_path)
    },
}
artifact_paths['manifest'].write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)

artifact_table = []
for name, path in {
    **model_paths, **artifact_paths,
    'target_metadata': target_path, 'target_schema': target_schema_path,
}.items():
    artifact_table.append({
        'artifact': name, 'path': str(path), 'size_mb': path.stat().st_size / 1024**2
    })
display(pd.DataFrame(artifact_table))
print('저장 완료')

## 8. 기존 target 모델과 동일 정답에서 비교

기존 ModernTCN 예측을 downside-aware target으로 다시 채점하고, 새 target으로 재학습한 모델과 비교한다. 이렇게 해야 target 난이도 변화와 재학습 효과를 분리할 수 있다.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score, log_loss, brier_score_loss

BASELINE_ROOT = (PROJECT_ROOT / '../../results/training/moderntcn_ohlc_60m_v1').resolve()
baseline_oof = pd.read_parquet(BASELINE_ROOT / 'oof_predictions.parquet')
baseline_test = pd.read_parquet(BASELINE_ROOT / 'test_predictions.parquet')


def comparison_metrics(frame, probability_column):
    y = frame['target_downside_aware_tp3_3m'].to_numpy()
    p = frame[probability_column].to_numpy()
    downside = frame['downside_3m'].to_numpy()
    mfe = frame['mfe_3m'].to_numpy()
    return {
        'samples': len(frame),
        'prevalence': float(y.mean()),
        'PR_AUC': float(average_precision_score(y, p)),
        'PR_lift': float(average_precision_score(y, p) / y.mean()),
        'ROC_AUC': float(roc_auc_score(y, p)),
        'log_loss': float(log_loss(y, p, labels=[0, 1])),
        'brier': float(brier_score_loss(y, p)),
        'p_vs_mfe_spearman': safe_spearman(mfe, p),
        'p_vs_downside_spearman': safe_spearman(downside, p),
        'p_vs_excursion_utility_spearman': safe_spearman(mfe - downside, p),
        'prediction_mean': float(p.mean()),
    }


comparison_rows = []
for phase, new_frame, baseline_frame in [
    ('OOF', oof_predictions, baseline_oof),
    ('Test', test_predictions, baseline_test),
]:
    baseline = new_frame[[
        'sample_id', 'target_downside_aware_tp3_3m', 'mfe_3m',
        'downside_3m',
    ]].merge(
        baseline_frame[['sample_id', 'pred_open_hard_probability']],
        on='sample_id', how='left', validate='one_to_one'
    )
    assert baseline['pred_open_hard_probability'].notna().all()
    comparison_rows.append({
        'model': '기존 original-target ModernTCN', 'phase': phase,
        **comparison_metrics(baseline, 'pred_open_hard_probability'),
    })
    comparison_rows.append({
        'model': '새 downside-aware ModernTCN', 'phase': phase,
        **comparison_metrics(new_frame, 'pred_downside_aware_probability'),
    })

comparison = pd.DataFrame(comparison_rows)
display(comparison)

delta = comparison.pivot(index='phase', columns='model', values=[
    'PR_AUC', 'ROC_AUC', 'p_vs_downside_spearman',
    'p_vs_excursion_utility_spearman',
])
display(delta)
print('판정 기준: PR/ROC가 오르고 downside 상관이 내려가며 excursion utility 상관이 올라야 target 변경이 성공입니다.')

In [ ]:
## 9. 상위 점수 구간의 downside 변화

전체 PR-AUC뿐 아니라 새 모델의 상위 점수 구간에서 양방향 고변동 false positive가 실제로 줄었는지 확인한다.

top_rows = []
for phase, new_frame, baseline_frame in [
    ('OOF', oof_predictions, baseline_oof),
    ('Test', test_predictions, baseline_test),
]:
    joined = new_frame.merge(
        baseline_frame[['sample_id', 'pred_open_hard_probability']],
        on='sample_id', how='left', validate='one_to_one'
    )
    for model_name, probability_column in [
        ('기존 original-target ModernTCN', 'pred_open_hard_probability'),
        ('새 downside-aware ModernTCN', 'pred_downside_aware_probability'),
    ]:
        threshold = joined[probability_column].quantile(0.95)
        selected = joined[joined[probability_column].ge(threshold)]
        false_positive = selected[selected['target_downside_aware_tp3_3m'].eq(0)]
        top_rows.append({
            'model': model_name,
            'phase': phase,
            'top_fraction': 0.05,
            'selected': len(selected),
            'precision': float(selected['target_downside_aware_tp3_3m'].mean()),
            'selected_downside_median': float(selected['downside_3m'].median()),
            'false_positive_downside_median': float(false_positive['downside_3m'].median()),
            'false_positive_downside_ge_3pct': float(
                false_positive['downside_3m'].ge(0.03).mean()
            ),
        })

top_score_comparison = pd.DataFrame(top_rows)
display(top_score_comparison)
print('다음 실험은 이 결과를 확정한 후, 같은 split에서 MFE-λ×downside utility를 직접 학습합니다.')